# Day 7: Correlation Analysis

In this notebook, we focus on identifying multi-collinearity among the numerical features in our dataset. Multi-collinearity can inflate the variance of coefficient estimates in linear models and make feature importance difficult to interpret in tree-based models. We will compute the Pearson correlation matrix and visualize it to highlight highly correlated feature pairs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Adjust path to import src modules
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_data_chunked

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 10)

## 1. Data Loading
We load the dataset using our memory-optimized data loader. We will then subset the numerical features for our analysis.

In [ ]:
dataset_path = r"C:\Users\siddp\Downloads\Dataset for default loan prediction\loan.csv"
df = load_data_chunked(dataset_path)
print(f"Dataset shape: {df.shape}")

## 2. Filtering Numerical Features
To compute the correlation matrix, we need to extract only the numerical columns. We'll also drop columns that are entirely NaN.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns
df_num = df[numerical_cols].dropna(axis=1, how='all')
print(f"Number of numerical features: {df_num.shape[1]}")

## 3. Computing and Visualizing the Correlation Matrix
With a large number of features, visualizing the entire matrix might be overwhelming, but a heatmap will help us quickly spot clusters of highly correlated features.

In [ ]:
# Compute Pearson correlation matrix
corr_matrix = df_num.corr()

# Create a mask for the upper triangle to make the plot cleaner
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(20, 18))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.title('Correlation Heatmap of Numerical Features', fontsize=18)
plt.tight_layout()
plt.show()

## 4. Identifying Highly Correlated Pairs
Let's programmatically extract pairs of features that have a correlation coefficient greater than 0.8 (or less than -0.8). These are prime candidates for removal or consolidation in our feature selection phase.

In [ ]:
# Flatten the matrix to easily sort and filter pairs
corr_pairs = corr_matrix.unstack()

# Remove self-correlations and duplicate pairs
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
sorted_pairs = corr_pairs.sort_values(kind="quicksort", ascending=False)

# Keep only one of each pair (since corr(A,B) == corr(B,A))
seen = set()
unique_highly_correlated = []
for (f1, f2), val in sorted_pairs.items():
    if (f2, f1) not in seen and abs(val) > 0.8:
        unique_highly_correlated.append(((f1, f2), val))
        seen.add((f1, f2))

print("Highly Correlated Feature Pairs (|corr| > 0.8):\n")
for (f1, f2), val in unique_highly_correlated:
    print(f"{f1:30} & {f2:30} : {val:.4f}")

## Conclusion
We found several sets of highly redundant features, such as:
* `loan_amnt`, `funded_amnt`, and `funded_amnt_inv` (Correlation ~1.0)
* `total_pymnt`, `total_pymnt_inv`, and `total_rec_prncp` (Correlation ~0.99)
* `open_acc` and `num_sats` (Correlation ~1.0)
* `loan_amnt` and `installment` (Correlation ~0.95)

In Phase 4 (Feature Engineering & Selection), we will drop the redundant counterparts to streamline our model.